In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3://bp-glue-demo/raw/market_prices")
df.display()

In [0]:
from pyspark.sql.functions import col, when, upper, expr, trim

#  Remove duplicates
df = df.dropDuplicates()

#  Remove null settlement_price
df = df.filter(col("settlement_price").isNotNull())

#  Standardize commodity
df = df.withColumn("commodity", upper(col("commodity")))

df = df.replace({
    "NAT_GAS": "NATURAL_GAS",
    "GAS-OIL": "GAS_OIL",
    "JET FUEL": "JET_FUEL"
}, subset=["commodity"])

# Fill null benchmark
df = df.withColumn(
    "benchmark",
    when(col("benchmark").isNull(), "UNKNOWN").otherwise(col("benchmark"))
)

# CRITICAL FIX: Safe date handling (NO ERRORS)
df = df.withColumn(
    "price_date_clean",
    expr("""
        coalesce(
            try_to_date(trim(price_date), 'yyyy-MM-dd'),
            try_to_date(trim(price_date), 'MM/dd/yyyy'),
            try_to_date(trim(price_date), 'dd/MM/yyyy')
        )
    """)
)

# Remove invalid dates
df = df.filter(col("price_date_clean").isNotNull())

# Replace old column with clean one
df = df.drop("price_date") \
       .withColumnRenamed("price_date_clean", "price_date")

#  Add flags
df = df.withColumn(
    "is_outlier",
    when(col("settlement_price") > 1000, True).otherwise(False)
)

df = df.withColumn(
    "is_negative",
    when(col("settlement_price") < 0, True).otherwise(False)
)


df.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable("market_prices_silver")

In [0]:
%sql
select * from market_prices_silver